In [8]:
import requests
import json
import time
import re
import pandas as pd
from datetime import datetime

start_time = datetime.now()
print("starting time : ", start_time)

output_data_template = {
    "InputBENEFICIARY_NAME" : None,
    "legalName": None,
    "gst": None,
    "pan": None,
}

def write_to_txt_resolve(output_data_template):
    file1 = r'D:\Nexensus_Projects\MasterIndia\output_folder\data4.txt'
    text_file = open(file1, 'a')
    text_file.write(output_data_template)
    text_file.write(",")
    text_file.write("\n")
    text_file.close()

def Master_India(name):
    try:
        updated_name_for_url = re.sub(r'\s+', ' ', name)
        url = "https://blog-backend.mastersindia.co/api/v1/custom/search/name_and_pan/?keyword=" + updated_name_for_url.replace(" ", "+")
        payload = {
            'keyword': updated_name_for_url
        }
        headers = {
            'Accept': 'application/json, text/plain, */*',
            'Accept-Encoding': 'gzip, deflate, br, zstd',
            'Accept-Language': 'en-US,en;q=0.9',
            'Origin': 'https://www.mastersindia.co',
            'Referer': 'https://www.mastersindia.co/gst-number-search-by-name-and-pan/',
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
        }

        delay = 5  # initial delay in seconds

        while True:
            response_post = requests.get(url, headers=headers, params=payload)
            
            if response_post.status_code == 429:
                retry_after = int(response_post.headers.get("Retry-After", delay))
                print(f"🔁 429 Too Many Requests for {name}. Retrying in {retry_after} seconds...")
                time.sleep(retry_after)
                continue

            break  # exit loop if not 429

        if response_post.status_code == 206:
            print("No data available for:", name)
        else:
            result = response_post.json()
            datas = result['data']
            print(datas)

            for i in datas:
                if updated_name_for_url.strip().title() == i['lgnm'].strip().title():
                    output_data_template['InputBENEFICIARY_NAME'] = name
                    output_data_template['legalName'] = i['lgnm']
                    output_data_template['gst'] = i['gstin']
                    output_data_template['pan'] = i['gstin'][2:12]
                    write_to_txt_resolve(json.dumps(output_data_template))
                    print("✅ Data stored for company:", name)

    except Exception as e:
        print("❌ Exception for company {}:".format(name), e)
        with open(r"D:\\Nexensus_Projects\\MasterIndia\\output_folder\\failedData4.txt", "a", encoding='utf-8') as f:
            f.write(name)
            f.write("\n")


try:
    df = pd.read_excel(r'D:\\Nexensus_Projects\\MasterIndia\\input_folder\\remaining_limited.xlsx')
    if 'BENEFICIARY_NAME' not in df.columns:
        raise KeyError("'BENEFICIARY_NAME' column not found in the Excel file.")

    # company_names = df['BENEFICIARY_NAME'].dropna().unique()
    company_names = df['Company'].dropna().unique()

    # Step 2: Process Each Company Name
    for company in company_names[:]:
        Master_India(str(company))
        time.sleep(5)
        # time.sleep(20)  # Adjust delay as needed to avoid rate limits

except Exception as e:
    print(f"❌ Error reading Excel file: {e}")

end_time = datetime.now()
print("starting time : ", end_time)

starting time :  2025-05-26 12:41:19.820550
No data available for: A B B INDIA LIMITED
No data available for: A K INFRADREAM LIMITED
No data available for: A AND J MICRONS PRIVATE LIMITED
No data available for: AADESHWAR PROHOUSE PRIVATE LIMITED
No data available for: ABC ELECTRICALS PRIVATE LIMITED
No data available for: ABHAY DEVCONS PRIVATE LIMITED
No data available for: ABHINAV HOMEDECOR PRIVATE LIMITED
No data available for: ABHINAV METALS PRIVATE LIMITED HDFC
[{'stjCd': 'GJ008', 'stj': 'Ghatak 8 (Ahmedabad)', 'lgnm': 'ACCESS POINT INDIA LIMITED', 'dty': 'Regular', 'adadr': [], 'cxdt': '', 'gstin': '24AATCA6112D1ZY', 'nba': ['Supplier of Services', 'Works Contract'], 'lstupdt': '28/02/2025', 'rgdt': '03/10/2020', 'ctb': 'Public Limited Company', 'pradr': {'addr': {'bnm': 'RAJA COMPLEX', 'loc': 'AHMEDABAD', 'st': 'NR VIJAY CROSS ROAD, NAVRANGPURA', 'bno': '5 TF', 'dst': 'Ahmedabad', 'lt': '', 'locality': '', 'pncd': '380009', 'landMark': '', 'stcd': 'Gujarat', 'geocodelvl': 'NA', '

In [ ]:
name = "A  K  INTERNATIONAL GUJARAT"
updated_name_for_url = re.sub(r'\s+', ' ', name)
url = "https://blog-backend.mastersindia.co/api/v1/custom/search/name_and_pan/?keyword=" + updated_name_for_url.replace(" ", "+")
payload = {
    'keyword': updated_name_for_url
}
headers = {
    'Accept':'application/json, text/plain, */*',
    'Accept-Encoding':'gzip, deflate, br, zstd',
    'Accept-Language':'en-US,en;q=0.9',
    'Origin':'https://www.mastersindia.co',
    'Referer':'https://www.mastersindia.co/gst-number-search-by-name-and-pan/',
    'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36'
}
response_post = requests.get(url, headers = headers, params=payload)
result = response_post.json()
# datas = result['data']
print(response_post)

response_post.status_code == 206

<Response [429]>


In [ ]:
import json

# Input and output file paths
input_file = r"D:\\Nexensus_Projects\\MasterIndia\\output_folder\\data2.txt"
output_file = "unique_companies.txt"

# Set to store unique legal names
seen_names = set()
unique_entries = []

# Read and process each line
with open(input_file, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip().rstrip(",")  # Clean up trailing comma
        if not line:
            continue
        try:
            entry = json.loads(line)
            name = entry.get("InputBENEFICIARY_NAME", "")
            if name not in seen_names:
                print(name)
                seen_names.add(name)
                unique_entries.append(entry)
        except json.JSONDecodeError:
            print("Skipping invalid line:", line)

# Write unique entries back to a file
with open(output_file, "w", encoding="utf-8") as f:
    for entry in unique_entries:
        json.dump(entry, f)
        f.write(",\n")

print(f"Done. Found {len(unique_entries)} unique entries.")


A  K  ENGINEERING
A  K  ENTERPRISES
A  N ASSOCIATES
A 1 INFRASTRUCTURE
A B CONSTRUCTION
A B JEWELLERS
A B MARKETING CO
A B Marketing
A D CONSTRUCTIONS
A D JEWELLERS
A G AGENCIES
A G AND CO
A G Engineering
A H ENTERPRISES
A I ENTERPRISES
A J Enterprises
A J INDUSTRIES
A J MARKETING
A J MILLS STORES
A J TEXTILES
A J TRADING CO
A K AGENCIES
A K AGRO TOBACCO TRADERS
A K ASSOCIATES
A K CONSTRUCTION
A K Constructions
A K DAS ASSOCIATES LIMITED
A K ENTERPRISES
A K INDUSTRIES
A K Jewellers
A K R CO
A K R FOOD PRODUCTS
A K SHIVHARE INFRASTRUCTURE
A K TRADERS
A K and  CO
A Kumar and co
A LOOKMANJEE AND CO
A M Associate
A M Constructions
A M ENTERPRISES
A M Engineering
A M Engineers
A M JEWELLERS
A M MODERN RICE MILL
A M REDDY AUTO AGENCIES
A M S TRADER
A M STEEL
A M STEEL TRADERS
A M TRADING COMPANY
A MAHENDRA JEWELLERS
A N ASSOCIATES
A N ENTERPRISES
A N TRADELINKS
A N TRADERS
A ONE CORPORATION
A ONE COTTON MILL
A ONE ENGINEERING
A ONE ENTERPRISES
A ONE FOOD INDUSTRY
A ONE FOOD PRODUCT
A ONE IND

In [5]:
import pandas as pd

# Load both Excel files
total_df = pd.read_excel(r'D:\Nexensus_Projects\MasterIndia\input_folder\List of entities.xlsx')         # Replace with your file path
completed_df = pd.read_excel(r'D:\Nexensus_Projects\MasterIndia\output_folder\CSVs\Master_India_Data.xlsx') # Replace with your file path
completed_df_2 = pd.read_csv(r'D:\Nexensus_Projects\MasterIndia\output_folder\CSVs\Master_India_Data_2.csv')
# Assuming the column to match is named 'Company'
# Make sure both columns are of string type and stripped
total_df['Company'] = total_df['BENEFICIARY_NAME'].astype(str).str.strip()
completed_df['Company'] = completed_df['InputBENEFICIARY_NAME'].astype(str).str.strip()
completed_df_2['Company'] = completed_df_2['InputBENEFICIARY_NAME'].astype(str).str.strip()

# Filter companies in total_df that are NOT in completed_df
remaining_df = total_df[~total_df['Company'].isin(completed_df['Company'])]
print(len(remaining_df))
remaining_df = remaining_df[~remaining_df['Company'].isin(completed_df_2['Company'])]

print(len(remaining_df))

# Save the result to a new Excel file
remaining_df.to_excel('remaining_companies.xlsx', index=False)

print("Remaining companies saved to 'remaining_companies.xlsx'")


329
259
Remaining companies saved to 'remaining_companies.xlsx'


In [10]:
import pandas as pd

# Load both Excel files
total_df = pd.read_excel(r'D:\Nexensus_Projects\MasterIndia\input_folder\remaining_limited.xlsx')         # Replace with your file path
# completed_df = pd.read_excel(r'D:\Nexensus_Projects\MasterIndia\output_folder\CSVs\Master_India_Data.xlsx') # Replace with your file path
completed_df_3 = pd.read_csv(r'D:\Nexensus_Projects\MasterIndia\output_folder\CSVs\Master_India_Data_3.csv')
# Assuming the column to match is named 'Company'
# Make sure both columns are of string type and stripped
total_df['Company'] = total_df['Company'].astype(str).str.strip()
# completed_df['Company'] = completed_df['InputBENEFICIARY_NAME'].astype(str).str.strip()
completed_df_3['Company'] = completed_df_3['InputBENEFICIARY_NAME'].astype(str).str.strip()

# Filter companies in total_df that are NOT in completed_df
remaining_df = total_df[~total_df['Company'].isin(completed_df_3['Company'])]
# print(len(remaining_df))
# remaining_df = remaining_df[~remaining_df['Company'].isin(completed_df_2['Company'])]

print(len(remaining_df))

# Save the result to a new Excel file
remaining_df.to_excel('remaining_limited_companies.xlsx', index=False)

print("Remaining companies saved to 'remaining_limited_companies.xlsx'")


60
Remaining companies saved to 'remaining_limited_companies.xlsx'


In [6]:
import pandas as pd
df = pd.read_excel(r'D:\\Nexensus_Projects\\MasterIndia\\input_folder\\remaining_companies.xlsx')
if 'BENEFICIARY_NAME' not in df.columns:
    raise KeyError("'BENEFICIARY_NAME' column not found in the Excel file.")

company_names = df['BENEFICIARY_NAME'].dropna().unique()

# Step 2: Process Each Company Name
for company in company_names[125:]:
    print(company)

A S ADVERTISING CO OD
A S Agro foods
A S BIRBITTE AND CO
A S Buildcon
A S CONSTRUCTION  COMPANY
A S CONSTRUCTION VAASAN
A S CONVEYORS
A S DAS AND SONS CO
A S ENGG WORKS
A S ENTERPRISES JAIPUR
A S INFRATECH
A S JAJOO CONTRATOR
A S JAKHAR COTTON GINN MILLS AXIS
A S MOOSANI AND COMPANY
A S MOOSANI AND COMPANY BMC
A S POLYCHEM INDIA  LLP
A S STEEL TRADERS VSP PL
A SANGAIAH AND CO
A SHANTHI SNATCHER TECHNOLOGIES
A SRI RAM AND SONS
A SUBRAMANIAM AND SONS
A T  CONSTRUCTIONS
A T AND CONSTRUCTION
A TO Z TRAVELLARS
A V FREIGHT SOLUTIONS
A V M TRADERS
A V N TRADE LINKS
A V THOMAS  CO LTD
A b rubber traders
A to Z Sports Infra LLP
A AND J MICRONS PRIVATE LIMITED
AADESHWAR PROHOUSE PRIVATE LIMITED
AADHIRA CNC PRODUCTS PRIVATE LIMITE
AARATHANA AGRO INDIA PRIVATE LIMITE
ABC ELECTRICALS PRIVATE LIMITED
ABHAY DEVCONS PRIVATE LIMITED
ABHIBHASYA PRIVATE LIMITE
ABHIMANYU ENTERPRISES PRIVATE LTD
ABHINAV HOMEDECOR PRIVATE LIMITED
ABHINAV METALS PRIVATE LIMITED HDFC
ABHISHEK PROPERTIES I PRIVATE
ABRECO MOTOR